In [ ]:
from training_env.market import Market
from models.algorithms.ppo import train, set_seed
import random
import os

In [ ]:
seed = 42
random.seed(seed)
set_seed(seed)

def train_test_split(data: list, test_size: float = 0.15) -> tuple[list, list]:
    random.shuffle(data)

    splitter = int(len(data) * test_size)

    return data[splitter:], data[:splitter]


files = []
path_to_train_data = '../data/train'
path_to_test_data = '../data/test'

train_files = []
validation_files = []
test_files = []

for file in os.listdir(path_to_train_data):
    files.append(os.path.join(path_to_train_data, file))

for file in os.listdir(path_to_test_data):
    test_files.append(os.path.join(path_to_test_data, file))

train_files, validation_files = train_test_split(files, test_size=0.04)

In [ ]:
print('Test', len(test_files))
print('Val', len(validation_files))
print('Train', len(train_files))
print(f'{len(files)}={len(test_files)}+{len(train_files)}+{len(validation_files)}')

In [ ]:
features = ['log_close', 'log_high', 'log_low', 'log_volume', 'log_return', 'sma20', 'rsi14', 'macd', 'signal', 'hist', 'close_raw']

train_env = Market(training_files=train_files, features=features, initial_cash=10_000, slippage=0.03, broker_fee=0.01, lam=0.1, seed=seed)
eval_env = Market(training_files=validation_files, features=features, initial_cash=10_000, slippage=0.03, broker_fee=0.01, lam=0.1, seed=seed)

In [ ]:
import optuna


def objective(trial: optuna.trial.Trial):
    hidden_layers = trial.suggest_int(name='hidden_layers', low=2, high=32, step=1)
    hidden_units = trial.suggest_int(name='hidden_units', low=32, high=128, step=16)

    actor_lr = trial.suggest_float(name='actor_lr', low=1e-5, high=3e-4, log=True)
    critic_lr = trial.suggest_float(name='critic_lr', low=1e-5, high=1e-4, log=True)

    gamma = trial.suggest_float(name='gamma', low=0.95, high=0.999)
    lam = trial.suggest_float(name='lam', low=0.9, high=0.99)

    opt_epochs = trial.suggest_int(name='opt_epochs', low=3, high=10)

    clip_ratio = trial.suggest_float(name='clip_ratio', low=0.1, high=0.3)
    grad_norm = trial.suggest_float(name='gradient clipping', low=0.3, high=2.0)

    c1 = trial.suggest_float(name='c1', low=0.1, high=0.7)
    c2 = trial.suggest_float(name='c2', low=1e-4, high=1e-2, log=True)

    batch_size = trial.suggest_int(name='batch_size', low=128, high=512, step=128)
    memory_size = trial.suggest_int(name='memory_size', low=2, high=10, step=1)

    params = {
        'env': train_env,
        'eval_env': eval_env,
        'hidden_layers': hidden_layers,
        'hidden_units': hidden_units,
        'memory_size': memory_size,
        'grad_norm': grad_norm,
        'actor_lr': actor_lr,
        'critic_lr': critic_lr,
        'advantage_type': 'gae',
        'gamma': gamma,
        'lam': lam,
        'clip_ratio': clip_ratio,
        'opt_epochs': opt_epochs,
        'c1': c1,
        'c2': c2,
        'batch_size': batch_size,
        'display_stat': False,
        'eval_episodes': 6,
        'total_steps': 50_000,
        'seed': seed,
        'debug': False,
        'del_model': True
    }

    rewards = train(**params)

    return rewards


study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=60)

In [ ]:
best_trial = study.best_trial

In [ ]:
print(f'Avg. Rewards per one episode {best_trial.value:.4f}')
for param in best_trial.params:
    print(f'\t{param} -> {best_trial.params[param]}')

In [ ]:
# New envs
train_files.extend(validation_files)
train_env = Market(training_files=train_files, features=features, initial_cash=10_000, slippage=0.03, broker_fee=0.01, lam=0.1, seed=seed)
test_env = Market(training_files=test_files, features=features, initial_cash=10_000, slippage=0.03, broker_fee=0.01, lam=0.1, seed=seed)

In [ ]:
best_trial.params['grad_norm'] = best_trial.params['gradient clipping']
del best_trial.params['gradient clipping']

best_trial.params

In [ ]:
train(
    env=train_env,
    eval_env=test_env,
    advantage_type='gae',
    total_steps=630_000*2,
    seed=seed,
	**best_trial.params
)